# Prepare Frame-Classification Annotation Samples

This notebook creates protected XLSX handoffs for human pilot and validation annotation, plus CSV pools for LLM annotation and full-context deployment. It consumes the shared LSC mention table and keeps all annotation samples disjoint and auditable.


## Setup

The sampling contract is fixed here so the human and LLM-labelled pools remain disjoint and auditable.

In [ ]:
from __future__ import annotations

import hashlib
from pathlib import Path

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Alignment, Font, PatternFill, Protection
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
HUMAN_DIR = OUTPUT_DIR / "human_annotation"
LLM_DIR = OUTPUT_DIR / "llm_annotation"

for path in [OUTPUT_DIR, HUMAN_DIR, LLM_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260604
PILOT_N = 200
VALIDATION_N = 400
LLM_TRAINING_N = 2000
CODEBOOK_VERSION = "v0.2"
OVERWRITE_EXISTING_ANNOTATION_HANDOFFS = False


## Load Target Contexts

Only ADHD and Autism contexts are frame-labelled. Baseline terms remain unframed comparators.

In [ ]:
required_columns = [
    "doc_id",
    "url",
    "registered_domain",
    "analysis_unit",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "target_sentence_plus_adjacent",
    "lsc_year",
    "source_year",
]

contexts = pd.read_parquet(CONTEXT_PATH, columns=required_columns)
target_contexts = contexts.loc[contexts["analysis_unit"].isin(["ADHD", "Autism"])].copy()
target_contexts = target_contexts.dropna(subset=["target_sentence_plus_adjacent", "lsc_year"])
target_contexts = target_contexts.loc[target_contexts["target_sentence_plus_adjacent"].str.strip().ne("")].copy()

def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]

target_contexts["context_id"] = target_contexts.apply(stable_context_id, axis=1)
target_contexts["year_band"] = pd.cut(
    target_contexts["lsc_year"].astype(int),
    bins=[2013, 2017, 2022, 2026],
    labels=["early_2014_2017", "mid_2018_2022", "late_2023_2026"],
)

duplicate_context_ids = target_contexts["context_id"].duplicated().sum()
if duplicate_context_ids:
    raise ValueError(f"Context ID collision or duplicate mention rows found: {duplicate_context_ids}")

target_contexts = target_contexts.sort_values(["analysis_unit", "lsc_year", "context_id"]).reset_index(drop=True)
print(f"Target contexts: {len(target_contexts):,}")
target_contexts.groupby(["analysis_unit", "year_band"], observed=True).size()

## Draw Disjoint Samples

Sampling is stratified by target group and broad year band. The LLM-training pool excludes the human pilot and validation rows.

In [ ]:
def stratified_sample(frame: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    group_columns = ["analysis_unit", "year_band"]
    groups = list(frame.groupby(group_columns, observed=True))
    if n > len(frame):
        raise ValueError(f"Requested {n} rows but only {len(frame)} are available")

    base = n // len(groups)
    remainder = n % len(groups)
    sampled_parts = []
    for index, (_, group) in enumerate(groups):
        group_n = min(len(group), base + (1 if index < remainder else 0))
        sampled_parts.append(group.sample(n=group_n, random_state=seed + index))

    sampled = pd.concat(sampled_parts, ignore_index=True)
    shortfall = n - len(sampled)
    if shortfall > 0:
        remaining = frame.loc[~frame["context_id"].isin(sampled["context_id"])]
        sampled = pd.concat([sampled, remaining.sample(n=shortfall, random_state=seed + 999)], ignore_index=True)
    return sampled.sample(frac=1, random_state=seed + 1000).reset_index(drop=True)

pilot = stratified_sample(target_contexts, PILOT_N, RANDOM_SEED)
remaining_after_pilot = target_contexts.loc[~target_contexts["context_id"].isin(pilot["context_id"])].copy()
validation = stratified_sample(remaining_after_pilot, VALIDATION_N, RANDOM_SEED + 10)
remaining_after_human = remaining_after_pilot.loc[~remaining_after_pilot["context_id"].isin(validation["context_id"])].copy()
llm_training = stratified_sample(remaining_after_human, LLM_TRAINING_N, RANDOM_SEED + 20)

assert set(pilot["context_id"]).isdisjoint(validation["context_id"])
assert set(pilot["context_id"]).isdisjoint(llm_training["context_id"])
assert set(validation["context_id"]).isdisjoint(llm_training["context_id"])

for name, sample in [("pilot", pilot), ("validation", validation), ("llm_training", llm_training)]:
    print(name, len(sample))
    print(sample.groupby(["analysis_unit", "year_band"], observed=True).size())

## Write Annotation Handoffs

Human annotation handoffs are protected XLSX workbooks. Context and metadata cells are stored explicitly as text and locked; the four annotation columns use constrained dropdowns. LLM and full-context pools remain machine-readable CSV files.


In [ ]:
annotation_columns = [
    "annotation_id",
    "context_id",
    "analysis_unit",
    "lsc_year",
    "raw_form",
    "target_sentence_plus_adjacent",
    "substantive_target_discourse",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "confidence",
    "annotation_round",
    "codebook_version",
]
editable_columns = [
    "substantive_target_discourse",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "confidence",
]


def human_sheet(sample: pd.DataFrame, prefix: str) -> pd.DataFrame:
    sheet = sample.copy().reset_index(drop=True)
    sheet["annotation_id"] = [f"{prefix}_{i:04d}" for i in range(1, len(sheet) + 1)]
    for column in editable_columns:
        sheet[column] = ""
    sheet["annotation_round"] = "pilot" if prefix == "pilot" else "validation"
    sheet["codebook_version"] = CODEBOOK_VERSION
    return sheet[annotation_columns]


def human_handoff_needs_refresh(path: Path) -> bool:
    if not path.exists():
        return True
    workbook = load_workbook(path, read_only=True, data_only=False)
    if "annotations" not in workbook.sheetnames:
        return True
    worksheet = workbook["annotations"]
    header = [cell.value for cell in worksheet[1]]
    if header != annotation_columns:
        return True
    version_column = annotation_columns.index("codebook_version") + 1
    versions = {
        str(worksheet.cell(row=row, column=version_column).value)
        for row in range(2, min(worksheet.max_row, 6) + 1)
        if worksheet.cell(row=row, column=version_column).value is not None
    }
    return versions != {CODEBOOK_VERSION}


def write_annotation_workbook(sheet: pd.DataFrame, path: Path) -> str:
    if path.exists() and not (human_handoff_needs_refresh(path) or OVERWRITE_EXISTING_ANNOTATION_HANDOFFS):
        return "kept existing"

    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "annotations"
    validation_sheet = workbook.create_sheet("validation_values")

    validation_values = {
        "A": ["TRUE", "FALSE"],
        "B": ["TRUE", "FALSE", "NA"],
        "C": ["high", "medium", "low"],
    }
    for column, values in validation_values.items():
        for row_number, value in enumerate(values, start=1):
            validation_sheet[f"{column}{row_number}"] = value
    validation_sheet.sheet_state = "hidden"

    header_fill = PatternFill("solid", fgColor="1F4E78")
    immutable_fill = PatternFill("solid", fgColor="E7EEF5")
    editable_fill = PatternFill("solid", fgColor="FFF2CC")
    invalid_fill = PatternFill("solid", fgColor="F4CCCC")

    for column_number, column_name in enumerate(annotation_columns, start=1):
        cell = worksheet.cell(row=1, column=column_number, value=column_name)
        cell.font = Font(color="FFFFFF", bold=True)
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.protection = Protection(locked=True)

    for row_number, row in enumerate(sheet.itertuples(index=False, name=None), start=2):
        for column_number, value in enumerate(row, start=1):
            column_name = annotation_columns[column_number - 1]
            cell = worksheet.cell(row=row_number, column=column_number)
            if value is not None and not pd.isna(value) and str(value) != "":
                cell.value = str(value)
                cell.data_type = "s"
            else:
                cell.value = None
            cell.number_format = "@"
            cell.alignment = Alignment(vertical="top", wrap_text=column_name == "target_sentence_plus_adjacent")
            if column_name in editable_columns:
                cell.fill = editable_fill
                cell.protection = Protection(locked=False)
            else:
                cell.fill = immutable_fill
                cell.protection = Protection(locked=True)
        worksheet.row_dimensions[row_number].height = 60

    validation_ranges = {
        "substantive_target_discourse": "'validation_values'!$A$1:$A$2",
        "clinical_frame_present": "'validation_values'!$B$1:$B$3",
        "lived_experience_frame_present": "'validation_values'!$B$1:$B$3",
        "confidence": "'validation_values'!$C$1:$C$3",
    }
    for column_name, formula in validation_ranges.items():
        column_letter = get_column_letter(annotation_columns.index(column_name) + 1)
        validation = DataValidation(type="list", formula1=formula, allow_blank=True)
        validation.error = "Select a value from the dropdown."
        validation.errorTitle = "Invalid annotation value"
        validation.showErrorMessage = True
        worksheet.add_data_validation(validation)
        validation.add(f"{column_letter}2:{column_letter}{len(sheet) + 1}")

    for error_value in ["#NAME?", "#VALUE!", "#REF!", "#DIV/0!", "#NUM!", "#NULL!", "#N/A", "#SPILL!", "#CALC!"]:
        worksheet.conditional_formatting.add(
            f"A2:L{len(sheet) + 1}",
            FormulaRule(formula=[f'ISNUMBER(SEARCH("{error_value}",A2))'], fill=invalid_fill),
        )

    widths = {
        "annotation_id": 16,
        "context_id": 18,
        "analysis_unit": 14,
        "lsc_year": 10,
        "raw_form": 20,
        "target_sentence_plus_adjacent": 100,
        "substantive_target_discourse": 25,
        "clinical_frame_present": 22,
        "lived_experience_frame_present": 28,
        "confidence": 14,
        "annotation_round": 18,
        "codebook_version": 18,
    }
    for column_number, column_name in enumerate(annotation_columns, start=1):
        worksheet.column_dimensions[get_column_letter(column_number)].width = widths[column_name]

    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = f"A1:L{len(sheet) + 1}"
    worksheet.sheet_view.showGridLines = False
    worksheet.protection.sheet = True
    worksheet.protection.autoFilter = False
    worksheet.protection.sort = False
    worksheet.protection.selectLockedCells = False
    worksheet.protection.selectUnlockedCells = False
    workbook.save(path)
    return "wrote"


def write_generated_pool(frame: pd.DataFrame, path: Path) -> str:
    if path.exists() and not OVERWRITE_EXISTING_ANNOTATION_HANDOFFS:
        return "kept existing"
    frame.to_csv(path, index=False)
    return "wrote"


pilot_sheet = human_sheet(pilot, "pilot")
validation_sheet = human_sheet(validation, "validation")

pilot_path = HUMAN_DIR / "frame_pilot_annotation_blank.xlsx"
validation_path = HUMAN_DIR / "frame_validation_annotation_blank.xlsx"
llm_pool_path = LLM_DIR / "frame_llm_training_pool.csv"
full_pool_path = OUTPUT_DIR / "frame_target_context_pool.csv"

write_results = {
    pilot_path: write_annotation_workbook(pilot_sheet, pilot_path),
    validation_path: write_annotation_workbook(validation_sheet, validation_path),
    llm_pool_path: write_generated_pool(llm_training, llm_pool_path),
    full_pool_path: write_generated_pool(target_contexts, full_pool_path),
}

print("Annotation handoffs:")
for path, action in write_results.items():
    print(f"- {action}: {path.relative_to(PROJECT_ROOT)}")
